In [1]:
import networkx as nx
import numpy as np
from tqdm import tqdm
from collections import defaultdict

In [2]:
import obonet
import networkx as nx

def download_and_load_cl_graph():
    """Downloads the Cell Ontology (CL) and returns it as a NetworkX DiGraph."""
    url = "http://purl.obolibrary.org/obo/cl.obo"  # Official CL OBO file
    cl_graph = obonet.read_obo(url)

    dag = nx.DiGraph(cl_graph)
    return dag

cl_graph = download_and_load_cl_graph()
print(f"Loaded Cell Ontology with {len(cl_graph.nodes())} nodes and {len(cl_graph.edges())} edges.")

Loaded Cell Ontology with 17053 nodes and 46510 edges.


In [4]:
import pandas as pd

In [28]:
cell_types = pd.read_excel('./cellgen_cell_type_harmonizer.xlsx')

In [29]:
your_cell_types = list(cell_types['Cell_Type'])

In [30]:
celltype_to_clid, unmapped = map_cell_types_to_cl_ids(your_cell_types, cl_graph)

print("Mapped:", celltype_to_clid)
print("Unmapped:", unmapped)

Mapped: {'alveolar type 1 fibroblast cell': 'CL:4028004', 'unknown': None, 'type II pneumocyte': None, 'B cell': 'CL:0000236', 'capillary endothelial cell': 'CL:0002144', 'CD4-positive, alpha-beta T cell': 'CL:0000624', 'alveolar type 2 fibroblast cell': None, 'alveolar macrophage': 'CL:0000583', 'endothelial cell of lymphatic vessel': 'CL:0002138', 'CD8-positive, alpha-beta T cell': 'CL:0000625', 'elicited macrophage': 'CL:0000861', 'lung pericyte': 'CL:0009089', 'pulmonary artery endothelial cell': 'CL:1001568', 'type I pneumocyte': None, 'lung macrophage': 'CL:1001603', 'tracheobronchial smooth muscle cell': 'CL:0019019', 'multi-ciliated epithelial cell': None, 'CD1c-positive myeloid dendritic cell': 'CL:0002399', 'natural killer cell': 'CL:0000623', 'nasal mucosa goblet cell': 'CL:0002480', 'ciliated columnar cell of tracheobronchial tree': None, 'non-classical monocyte': 'CL:0000875', 'classical monocyte': 'CL:0000860', 'vein endothelial cell': 'CL:0002543', 'bronchus fibroblast o

In [36]:
import obonet
import networkx as nx
import numpy as np
from tqdm import tqdm
from collections import defaultdict

# 1. Download and load Cell Ontology
def download_and_load_cl_graph():
    """Downloads the Cell Ontology (CL) and returns it as a NetworkX DiGraph."""
    url = "http://purl.obolibrary.org/obo/cl.obo"
    cl_graph = obonet.read_obo(url)
    return nx.DiGraph(cl_graph)

# 2. Map cell types to CL IDs
def map_cell_types_to_cl_ids(cell_types, cl_graph):
    """Maps cell type names to CL IDs."""
    cl_id_to_name = {}
    for node_id, data in cl_graph.nodes(data=True):
        name = data.get("name", "").lower()
        synonyms = [syn.lower() for syn in data.get("synonym", [])]
        cl_id_to_name[node_id] = {"name": name, "synonyms": synonyms}

    celltype_to_clid = {}
    unmapped = []
    for ct in cell_types:
        ct_lower = ct.lower()
        matched = False
        for cl_id, names in cl_id_to_name.items():
            if (ct_lower == names["name"]) or (ct_lower in names["synonyms"]):
                celltype_to_clid[ct] = cl_id
                matched = True
                break
        if not matched:
            unmapped.append(ct)
            celltype_to_clid[ct] = None
    return celltype_to_clid, unmapped

# 3. Compute ancestry matrix
def get_ancestry_matrix(cl_graph, celltype_to_clid):
    """Computes ancestry relationships between cell types."""
    cell_types = list(celltype_to_clid.keys())
    num_celltypes = len(cell_types)
    ancestry = np.zeros((num_celltypes, num_celltypes), dtype=int)
    
    for i, ct_i in enumerate(cell_types):
        cl_id_i = celltype_to_clid[ct_i]
        if cl_id_i is None:
            continue
            
        for j, ct_j in enumerate(cell_types):
            cl_id_j = celltype_to_clid[ct_j]
            if cl_id_j is None:
                continue
                
            if nx.has_path(cl_graph, cl_id_j, cl_id_i):
                ancestry[i][j] = 1
    return ancestry

# 4. Compute PPR similarity
def get_ppr_similarity(cl_graph, celltype_to_clid, alpha=0.9, threshold=1e-6):
    """Computes PPR-based similarity between cell types."""
    cell_types = list(celltype_to_clid.keys())
    num_celltypes = len(cell_types)
    similarity = np.zeros((num_celltypes, num_celltypes))
    
    for i, ct_i in enumerate(tqdm(cell_types, desc="Computing PPR")):
        cl_id_i = celltype_to_clid[ct_i]
        if cl_id_i is None:
            continue
            
        ppr = nx.pagerank(cl_graph, alpha=alpha, personalization={cl_id_i: 1})
        
        for j, ct_j in enumerate(cell_types):
            cl_id_j = celltype_to_clid[ct_j]
            if cl_id_j is None:
                continue
                
            sim = ppr.get(cl_id_j, 0)
            if sim >= threshold:
                similarity[i][j] = np.log2(sim * (1 / threshold) + 1)
    return similarity

# 5. Harmonize cell types
def harmonize_cell_types(cell_types, ancestry, similarity, threshold=0.5):
    """Groups cell types based on ancestry and similarity."""
    groups = defaultdict(list)
    used = set()
    
    for i, ct_i in enumerate(cell_types):
        if ct_i in used:
            continue
            
        group = [ct_i]
        for j, ct_j in enumerate(cell_types):
            if i != j and (ancestry[i][j] == 1 or similarity[i][j] > threshold):
                group.append(ct_j)
                used.add(ct_j)
                
        groups[f"group_{i}"] = group
    return groups

# Main workflow
def main():
    # 1. Load CL graph
    cl_graph = download_and_load_cl_graph()
    
    # 2. Your cell types (example)
    # your_cell_types = ["T cell", "B cell", "neuron", "unknown_cell"]
    
    # 3. Map to CL IDs
    celltype_to_clid, unmapped = map_cell_types_to_cl_ids(your_cell_types, cl_graph)
    print("Unmapped cell types:", unmapped)
    
    # 4. Compute relationships
    ancestry = get_ancestry_matrix(cl_graph, celltype_to_clid)
    similarity = get_ppr_similarity(cl_graph, celltype_to_clid)
    
    # 5. Harmonize
    harmonized_groups = harmonize_cell_types(your_cell_types, ancestry, similarity)
    print("\nHarmonized groups:")
    for group, members in harmonized_groups.items():
        print(f"{group}: {members}")

if __name__ == "__main__":
    main()

Unmapped cell types: ['unknown', 'type II pneumocyte', 'alveolar type 2 fibroblast cell', 'type I pneumocyte', 'multi-ciliated epithelial cell', 'ciliated columnar cell of tracheobronchial tree', 'lung neuroendocrine cell', 'epithelial cell of alveolus of lung', 'respiratory hillock cell', 'brush cell of trachebronchial tree', 'brush cell', 'neuronal brush cell', 'SOX9_functionalis_II', 'eStromal', 'eStromal_MMPs', 'preGlandular', 'Immune_Myeloid', 'Glandular', 'SOX9_luminal', 'Cycling', 'eStromal_cycling', 'dStromal_mid', 'dStromal_early', 'Immune_Lymphoid', 'HOXA13', 'dStromal_late', 'MUC5B', 'eHormones', 'Venous', 'Glandular_secretory', 'Luminal', 'SOX9_basalis', 'preLuminal', 'preCiliated', 'ePV_1b', 'ePV_2', 'uSMCs', 'Arterial', 'Lymphatic', 'SOX9_functionalis_I', 'ePV_1a', 'Fibroblast_basalis', 'enterocyte of colon', 'lung ciliated cell', 'sHormones', 'KRT5', 'mPV', 'Glandular_secretory_FGF7', 'dHormones', 'differentiation-committed oligodendrocyte precursor']


Computing PPR: 100%|█████████████████████████████████████████████████████████████████| 370/370 [01:25<00:00,  4.33it/s]


Harmonized groups:
group_0: ['alveolar type 1 fibroblast cell', 'T cell', 'fibroblast', 'stromal cell', 'hematopoietic stem cell', 'hematopoietic precursor cell', 'neuron', 'endothelial cell', 'progenitor cell', 'lymphocyte', 'stem cell', 'epithelial cell', 'photoreceptor cell', 'ciliated epithelial cell', 'hematopoietic cell', 'neural cell', 'ciliated cell', 'mesodermal cell', 'common lymphoid progenitor', 'leukocyte']
group_1: ['unknown']
group_2: ['type II pneumocyte']
group_3: ['B cell', 'plasma cell', 'stromal cell', 'naive B cell', 'memory B cell', 'plasmablast', 'hematopoietic stem cell', 'mature B cell', 'hematopoietic precursor cell', 'neuron', 'endothelial cell', 'progenitor cell', 'transitional stage B cell', 'lymphoid lineage restricted progenitor cell', 'lymphocyte', 'stem cell', 'epithelial cell', 'hematopoietic cell', 'neural cell', 'T follicular helper cell', 'germinal center B cell', 'IgM plasma cell', 'IgA plasma cell', 'IgG plasma cell', 'precursor B cell', 'mesoder

In [37]:
cl_graph = download_and_load_cl_graph()
celltype_to_clid, unmapped = map_cell_types_to_cl_ids(your_cell_types, cl_graph)
ancestry = get_ancestry_matrix(cl_graph, celltype_to_clid)
similarity = get_ppr_similarity(cl_graph, celltype_to_clid)
harmonized_groups = harmonize_cell_types(your_cell_types, ancestry, similarity)

print("Unmapped types:", unmapped)
print("Harmonized groups:", harmonized_groups)


Computing PPR: 100%|█████████████████████████████████████████████████████████████████| 370/370 [01:30<00:00,  4.07it/s]

Unmapped types: ['unknown', 'type II pneumocyte', 'alveolar type 2 fibroblast cell', 'type I pneumocyte', 'multi-ciliated epithelial cell', 'ciliated columnar cell of tracheobronchial tree', 'lung neuroendocrine cell', 'epithelial cell of alveolus of lung', 'respiratory hillock cell', 'brush cell of trachebronchial tree', 'brush cell', 'neuronal brush cell', 'SOX9_functionalis_II', 'eStromal', 'eStromal_MMPs', 'preGlandular', 'Immune_Myeloid', 'Glandular', 'SOX9_luminal', 'Cycling', 'eStromal_cycling', 'dStromal_mid', 'dStromal_early', 'Immune_Lymphoid', 'HOXA13', 'dStromal_late', 'MUC5B', 'eHormones', 'Venous', 'Glandular_secretory', 'Luminal', 'SOX9_basalis', 'preLuminal', 'preCiliated', 'ePV_1b', 'ePV_2', 'uSMCs', 'Arterial', 'Lymphatic', 'SOX9_functionalis_I', 'ePV_1a', 'Fibroblast_basalis', 'enterocyte of colon', 'lung ciliated cell', 'sHormones', 'KRT5', 'mPV', 'Glandular_secretory_FGF7', 'dHormones', 'differentiation-committed oligodendrocyte precursor']
Harmonized groups: defau

In [41]:
def assign_unique_groups(harmonized_groups):
    """Ensure each cell type appears in only one group."""
    cell_to_group = {}
    for group_name, members in harmonized_groups.items():
        for cell in members:
            if cell not in cell_to_group:
                cell_to_group[cell] = group_name
    
    unique_groups = defaultdict(list)
    for cell, group in cell_to_group.items():
        unique_groups[group].append(cell)
    
    return dict(unique_groups)

In [42]:
fixed_groups = assign_unique_groups(harmonized_groups)

In [44]:
def label_by_highest_ancestor(group, cl_graph):
    """Picks the most general term that covers all group members."""
    common_ancestors = None
    for cell in group:
        if cell in cl_graph:
            ancestors = nx.ancestors(cl_graph, cell) | {cell}
            common_ancestors = ancestors if common_ancestors is None else common_ancestors & ancestors
    return next(iter(common_ancestors)) if common_ancestors else group[0]

In [46]:
labeled_groups = {}
for group_name, members in fixed_groups.items():
    label = label_by_highest_ancestor(members, cl_graph)  
    
    labeled_groups[group_name] = {
        "label": label,
        "members": members,
        "size": len(members)
    }

for group, data in labeled_groups.items():
    print(f"{group}: {data['label']} (covers {data['size']} cells)")

group_0: alveolar type 1 fibroblast cell (covers 20 cells)
group_1: unknown (covers 1 cells)
group_2: type II pneumocyte (covers 1 cells)
group_3: B cell (covers 22 cells)
group_4: capillary endothelial cell (covers 8 cells)
group_5: CD4-positive, alpha-beta T cell (covers 17 cells)
group_6: alveolar type 2 fibroblast cell (covers 1 cells)
group_7: alveolar macrophage (covers 10 cells)
group_8: endothelial cell of lymphatic vessel (covers 4 cells)
group_9: CD8-positive, alpha-beta T cell (covers 10 cells)
group_10: elicited macrophage (covers 5 cells)
group_11: lung pericyte (covers 3 cells)
group_12: pulmonary artery endothelial cell (covers 2 cells)
group_13: type I pneumocyte (covers 1 cells)
group_15: tracheobronchial smooth muscle cell (covers 3 cells)
group_16: multi-ciliated epithelial cell (covers 1 cells)
group_17: CD1c-positive myeloid dendritic cell (covers 1 cells)
group_18: natural killer cell (covers 5 cells)
group_19: nasal mucosa goblet cell (covers 5 cells)
group_20: c